In [1]:
pip install pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [1]:
import re
from pathlib import Path

import pandas as pd

INPUT = Path("asrs_combined.csv")
CLEAN_OUTPUT = Path("asrs_clean.csv")
CHUNKS_OUTPUT = Path("asrs_chunks.csv")

MIN_WORDS = 20
CHUNK_WORDS = 200

# Expand uppercase ASRS shorthand while preserving ordinary words.
ABBREV = {
    "ATC": "air traffic control",
    "TWR": "tower",
    "APCH": "approach",
    "ACFT": "aircraft",
    "RWY": "runway",
    "TWY": "taxiway",
    "CAPT": "captain",
    "F/O": "first officer",
    "FO": "first officer",
    "RA": "resolution advisory",
    "TA": "traffic advisory",
    "MEL": "minimum equipment list",
    "PIC": "pilot in command",
    "FMS": "flight management system",
    "ILS": "instrument landing system",
    "VFR": "visual flight rules",
    "IFR": "instrument flight rules",
}

# Match longer abbreviations first, so overlapping terms are handled safely.
abbrev_pattern = re.compile(
    r"(?<![A-Za-z/])("
    + "|".join(re.escape(term) for term in sorted(ABBREV, key=len, reverse=True))
    + r")(?![A-Za-z/])"
)

# Replace ASRS hidden-name tokens such as ZZZ, ZZZ1, XXX, or XXXX.
redacted_pattern = re.compile(r"\b(?:Z{2,}\d*|X{3,})\b", flags=re.IGNORECASE)


def find_column(columns, *keywords):
    """Find the first column whose normalized name contains every keyword."""
    for column in columns:
        name = str(column).strip().lower()
        if all(keyword in name for keyword in keywords):
            return column
    return None


def clean_text(value):
    text = str(value)
    text = abbrev_pattern.sub(lambda match: ABBREV[match.group(1)], text)
    text = redacted_pattern.sub("redacted", text)
    return re.sub(r"\s+", " ", text).strip().lower()


def split_long_text(text, max_words=CHUNK_WORDS):
    """Split on sentence boundaries where possible, and split oversized sentences."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks = []
    current_words = []

    for sentence in sentences:
        words = sentence.split()

        # Split a single sentence that is longer than the chunk target.
        sentence_parts = [
            words[i : i + max_words]
            for i in range(0, len(words), max_words)
        ]

        for part in sentence_parts:
            if not part:
                continue

            if current_words and len(current_words) + len(part) > max_words:
                chunks.append(" ".join(current_words))
                current_words = []

            current_words.extend(part)

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


if not INPUT.exists():
    raise FileNotFoundError(
        f"Could not find {INPUT.resolve()}. Put the CSV beside the notebook "
        "or change INPUT to its full file path."
    )

df = pd.read_csv(INPUT, dtype=str).fillna("")

print("Rows loaded:", len(df))
print("\nAll columns:")
for index, column in enumerate(df.columns):
    print(f"{index:3d}  {column}")

narrative_col = find_column(df.columns, "narrative")
synopsis_col = find_column(df.columns, "synopsis")
acn_col = find_column(df.columns, "acn")

if narrative_col is None:
    raise SystemExit("\nNo narrative column found. Check the column list above.")

print(
    f"\nUsing narrative column: {narrative_col}"
    f" | synopsis: {synopsis_col}"
    f" | ID: {acn_col}"
)

df["report_id"] = (
    df[acn_col].astype(str)
    if acn_col is not None
    else df.index.astype(str)
)
df["text"] = df[narrative_col].map(clean_text)

if synopsis_col is not None:
    df["synopsis_clean"] = df[synopsis_col].map(clean_text)

# Drop reports with fewer than 20 words.
before_short_filter = len(df)
df["n_words"] = df["text"].str.split().str.len()
df = df[df["n_words"] >= MIN_WORDS].copy()
dropped_short = before_short_filter - len(df)

# Drop repeated cleaned narratives. This keeps the first matching report.
before_deduplication = len(df)
df = df.drop_duplicates(subset="text").reset_index(drop=True)
dropped_duplicates = before_deduplication - len(df)

print(
    f"\nDropped {dropped_short:,} short reports and "
    f"{dropped_duplicates:,} duplicate narratives."
)
print(f"Reports kept: {len(df):,}")
print(f"Total dropped: {dropped_short + dropped_duplicates:,}")

# Narrative length summary and share over the chunk target.
print("\nNarrative length in words:")
print(df["n_words"].describe(percentiles=[0.5, 0.9, 0.95]).round(1))

long_share = (df["n_words"] > CHUNK_WORDS).mean() if len(df) else 0
print(f"Share longer than {CHUNK_WORDS} words: {long_share:.1%}")

# Make chunks and keep their report ID and order.
chunk_rows = []
for report_id, text in zip(df["report_id"], df["text"]):
    for chunk_number, chunk_text in enumerate(split_long_text(text)):
        chunk_rows.append({
            "report_id": report_id,
            "chunk_no": chunk_number,
            "text": chunk_text,
        })

chunks = pd.DataFrame(chunk_rows, columns=["report_id", "chunk_no", "text"])
print(f"\nChunks created: {len(chunks):,} from {len(df):,} reports")

# Save both output files in the notebook's current folder.
df.to_csv(CLEAN_OUTPUT, index=False, encoding="utf-8-sig")
chunks.to_csv(CHUNKS_OUTPUT, index=False, encoding="utf-8-sig")

print(f"\nSaved: {CLEAN_OUTPUT.resolve()}")
print(f"Saved: {CHUNKS_OUTPUT.resolve()}")

print("\nTwo cleaned examples:")
for text in df["text"].head(2):
    print("-", text[:400], "...\n") 

Rows loaded: 6372

All columns:
  0  acn
  1  date
  2  local_time_of_day
  3  locale_reference
  4  state_reference
  5  relative_positionangleradial
  6  relative_positiondistancenautical_miles
  7  altitudeaglsingle_value
  8  altitudemslsingle_value
  9  latitude__longitude_uas
 10  flight_conditions
 11  weather_elements__visibility
 12  work_environment_factor
 13  light
 14  ceiling
 15  rvrsingle_value
 16  atc__advisory
 17  aircraft_operator
 18  make_model_name
 19  aircraft_zone
 20  crew_size
 21  operating_under_far_part
 22  flight_plan
 23  mission
 24  nav_in_use
 25  flight_phase
 26  route_in_use
 27  airspace
 28  maintenance_statusmaintenance_deferred
 29  maintenance_statusrecords_complete
 30  maintenance_statusreleased_for_service
 31  maintenance_statusrequired__correct_doc_on_board
 32  maintenance_statusmaintenance_type
 33  maintenance_statusmaintenance_items_involved
 34  cabin_lighting
 35  number_of_seatsnumber
 36  passengers_on_boardnumber
 37  crew_siz

In [2]:
python -c "import pandas as pd; d=pd.read_csv('asrs_clean.csv',dtype=str).fillna(''); print((d['narrative1']!='').sum()); print(d.loc[d['narrative1']!='','narrative1'].head(3).str[:200].tolist())"

SyntaxError: invalid syntax (3768797565.py, line 1)